# 13 --- UMLS Ontology-Guided Query Expansion (on the MedQA benchmark)

**Idea: ground the query-expansion step in a medical ontology (UMLS) instead of a free-form LLM.**
In the earlier notebooks the cross-lingual gap was bridged by asking the LLM to translate each English
question into German and invent a few clinical synonyms. In this notebook I replace that step with the
**UMLS Metathesaurus**: each question is mapped to its UMLS concepts, and the search query is expanded with
those concepts' **official German synonyms** taken from the ontology. The hypothesis is that controlled
ontology vocabulary matches the AWMF guideline wording more reliably than LLM-generated synonyms, so
retrieval should improve.

Compared head to head, on the same MedQA benchmark and the same `awmf_baseline_bge` collection:
- **Baseline** = translate + MeSH-style query expansion (the method from the previous notebooks).
- **UMLS** = link the question to UMLS concepts, expand the query with their German synonyms.

Results are reported on the corpus-answerable subset, against **both** the corpus-grounded and the MedQA
ground truths, with the 4 RAGAS metrics. The Neon reconnect, judge `max_tokens`, and resumable-run fixes
from the earlier notebooks are kept. Single generator = Mistral. Nothing in the shared collection is modified.

**Access note:** UMLS needs a free licence. I create a UTS account at
https://uts.nlm.nih.gov/uts/signup-login and copy the API key into the Colab Secrets panel as `UMLS_API_KEY`.
The UTS REST API is used directly (only `requests`), which avoids the heavy `scispacy`/`nmslib` install that
breaks on recent Python versions.

## 1. Install

In [ ]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers sqlalchemy requests spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 69

## 2. VertexAI import patch

In [ ]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
m = types.ModuleType("langchain_community.llms"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms"] = m
m = types.ModuleType("langchain_community.chat_models"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models"] = m
m = types.ModuleType("langchain_community.chat_models.vertexai"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models.vertexai"] = m
m = types.ModuleType("langchain_community.llms.vertexai"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms.vertexai"] = m

## 3. Setup: resilient DB engine, embedder, reranker, models, prompts (same as notebooks 11/12)

In [ ]:
import os, json, time, random
import pandas as pd, torch, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder

nest_asyncio.apply()
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
df = pd.read_csv(DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv')
print("Loaded", len(df), "rows.")

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# Resilient Neon engine (serverless drops idle SSL connections)
engine = create_engine(NEON, pool_pre_ping=True, pool_recycle=180, pool_size=2, max_overflow=2,
                       connect_args={"sslmode":"require","keepalives":1,"keepalives_idle":30,
                                     "keepalives_interval":10,"keepalives_count":5})

bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vs = PGVector(embeddings=bge, collection_name="awmf_baseline_bge", connection=engine, use_jsonb=True)

K_RETRIEVE = 30
K_FINAL = 10           # the chosen k, used across all the notebooks (kept fixed here)
USE_RERANKER = False   # kept OFF on purpose: this experiment isolates whether UMLS ITSELF helps,
                       # with no reranker on top to confound the comparison
retriever = vs.as_retriever(search_kwargs={"k": K_RETRIEVE})

reranker = None
if USE_RERANKER:
    reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512,
                            device="cuda" if torch.cuda.is_available() else "cpu")
    print("reranker loaded")

def make_llm(model, max_tokens=1024):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            w = min(base*(2**a)+random.uniform(0,3), 120)
            print(f"  retry {a+1}: {str(e)[:70]} ... {w:.0f}s"); time.sleep(w)

# Baseline expansion: translate + MeSH synonyms (the method from the previous notebooks)
expand_prompt = PromptTemplate(template="""You are a medical search term generator.
Translate the English question to German, then add 3-4 formal German clinical synonyms / related conditions / MeSH terms.
Output ONLY the German question + synonyms as one continuous search string. No labels, no bullets.

English Question:
{question}""", input_variables=["question"])

qa_prompt = PromptTemplate(template="""You are an expert medical AI. Read the German clinical guidelines and answer the medical question in ENGLISH.
Use ONLY the provided German context. If the context does not contain the answer, say so plainly.

Context (German):
{context}

Question (English):
{question}

Answer (English):""", input_variables=["context","question"])

print("Setup complete.")

Mounted at /content/drive
Loaded 200 rows.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Setup complete.


## 4. UMLS concept linking + German-synonym expansion (UTS REST API)

The pipeline is light on dependencies: a small English spaCy model pulls candidate medical phrases from the
question, each phrase is matched to a UMLS concept (CUI) through the UTS search endpoint, and the concept's
**German atoms** are fetched as expansion terms. Calls are cached so each concept is looked up only once.
If the ontology returns nothing for a question, the method falls back to the baseline expansion so retrieval
never runs on an empty query.

In [ ]:
import requests, functools
import spacy

# Small English model just for candidate-phrase extraction (clean install, no nmslib).
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download as _dl; _dl("en_core_web_sm"); nlp = spacy.load("en_core_web_sm")

# Free UMLS key from a UTS account (https://uts.nlm.nih.gov/uts/signup-login), stored as a Colab secret.
try:
    UMLS_API_KEY = userdata.get('UMLS_API_KEY')
except Exception:
    UMLS_API_KEY = ''
if not UMLS_API_KEY:
    print("WARNING: UMLS_API_KEY is not set -> the UMLS method has no ontology to query.")
    print("Add it via the Colab Secrets panel (key icon) as UMLS_API_KEY, then re-run this cell.")

UTS = "https://uts-ws.nlm.nih.gov/rest"
# Generic words that are not useful concepts on their own.
GENERIC = {"patient","patients","treatment","diagnosis","management","therapy","risk","cause",
           "causes","symptom","symptoms","sign","signs","disease","condition","prognosis","feature","features"}

def candidate_phrases(question, max_n=6):
    doc = nlp(question)
    out, seen = [], set()
    for ch in doc.noun_chunks:
        t = " ".join(w.text for w in ch if not w.is_stop and not w.is_punct).strip()
        tl = t.lower()
        if len(tl) < 3 or tl in GENERIC or tl in seen: continue
        seen.add(tl); out.append(t)
        if len(out) >= max_n: break
    return out

@functools.lru_cache(maxsize=4096)
def search_cui(term):
    """Map a phrase to its best UMLS concept (CUI, English name)."""
    try:
        r = requests.get(f"{UTS}/search/current",
                         params={"string": term, "apiKey": UMLS_API_KEY, "pageSize": 1, "searchType": "words"},
                         timeout=20)
        hits = r.json().get("result", {}).get("results", []) if r.ok else []
        if hits and hits[0].get("ui") not in (None, "NONE"):
            return hits[0]["ui"], hits[0].get("name", term)
    except Exception:
        pass
    return None

@functools.lru_cache(maxsize=4096)
def german_atoms(cui):
    """German synonyms (atom names) of a concept from UMLS."""
    try:
        r = requests.get(f"{UTS}/content/current/CUI/{cui}/atoms",
                         params={"language": "GER", "apiKey": UMLS_API_KEY, "pageSize": 50}, timeout=20)
        if not r.ok: return ()
        names = []
        for a in r.json().get("result", []):
            n = (a.get("name") or "").strip()
            if n and all(n.lower() != x.lower() for x in names): names.append(n)
        return tuple(names[:6])
    except Exception:
        return ()

def expand_umls(question):
    de_terms, en_names = [], []
    for ph in candidate_phrases(question):
        hit = search_cui(ph)
        if not hit: continue
        cui, en = hit
        de = german_atoms(cui)
        if de: de_terms.extend(de)        # German ontology synonyms
        else:  en_names.append(en)         # no German atom -> keep the English concept name
    expanded = list(dict.fromkeys(de_terms + en_names))   # dedupe, German first
    if not expanded:
        return safe_invoke(mistral, expand_prompt.format(question=question))   # nothing linked -> baseline
    # Keep the English question as the cross-lingual anchor (BGE-M3 is multilingual), plus ontology terms.
    return question + " " + " ".join(expanded)

# smoke test
_q = df.iloc[0]['English_Open_Question']
print("Q:", _q)
print("Candidate phrases:", candidate_phrases(_q))
print("UMLS-expanded query:", expand_umls(_q)[:220], "...")

Q: A 58-year-old man comes to the emergency department because of increasing shortness of breath and a nonproductive cough for the last week. Three weeks ago, he had a fever and a cough for 6 days after he returned from a trip to Southeast Asia. He has had a 4-kg (9-lb) weight loss over the past 3 months. He has bronchial asthma and hypertension. He has smoked one pack of cigarettes daily for 41 years. Current medications include an albuterol inhaler and enalapril. His temperature is 37.6°C (99.7°F), pulse is 88/min, respirations are 20/min, and blood pressure is 136/89 mm Hg. There is dullness to percussion, decreased breath sounds, and decreased tactile fremitus over the left lung base. Cardiac examination shows no abnormalities. Chest x-ray of this patient is most likely to show What?
Candidate phrases: ['58 year old man', 'emergency department', 'increasing shortness', 'breath', 'nonproductive cough', 'week']
UMLS-expanded query: A 58-year-old man comes to the emergency department 

## 5. Retrieval functions: baseline expansion vs. UMLS expansion

In [ ]:
def _rerank_or_top(query, texts, k_final):
    if reranker is not None and texts:
        scores = reranker.predict([[query, t] for t in texts])
        texts = [t for _, t in sorted(zip(scores, texts), key=lambda x: x[0], reverse=True)]
    return texts[:k_final]

def retrieve_baseline(question, k_final=K_FINAL):
    gq = safe_invoke(mistral, expand_prompt.format(question=question))   # LLM translate + synonyms
    docs = retriever.invoke(gq)
    return _rerank_or_top(gq, [d.page_content for d in docs], k_final)

def retrieve_umls(question, k_final=K_FINAL):
    gq = expand_umls(question)                                          # ontology-driven expansion
    docs = retriever.invoke(gq)
    return _rerank_or_top(gq, [d.page_content for d in docs], k_final)

# smoke test
print("baseline top:", retrieve_baseline(_q)[0][:160], "...")
print("umls     top:", retrieve_umls(_q)[0][:160], "...")

baseline top: logische Veränderung der Lungenfunktion 
Rhinitis, akut rezidivierende/chronische Sinusitis, Postnasal-Drip-
Syndrom 
Haltungs- oder Nahrungsmittelabhängige 
Sy ...
umls     top: logische Veränderung der Lungenfunktion 
Rhinitis, akut rezidivierende/chronische Sinusitis, Postnasal-Drip-
Syndrom 
Haltungs- oder Nahrungsmittelabhängige 
Sy ...


## 6. Generate UMLS answers for all 200 (baseline reused from notebook 11)

Only **UMLS** is generated here, on **all 200 MedQA questions** (resumable). The baseline is **not** regenerated:
its answers are loaded from notebook 11's results file, produced with the same k, the same reranker setting,
and the same QA prompt over the same 200 questions -- so reusing it saves OpenRouter calls and keeps a single
canonical baseline shared by notebooks 11, 12 and 13. Both ground truths are stored per question, so the same
answers are scored two ways in the next section.

This notebook keeps the **reranker off** (to isolate UMLS's own effect), so the baseline reused here is
notebook 11's **no-reranker** run.

**Requirement:** notebook 11 must have finished its full 200-question run at `DEV=False` with the matching
`USE_RERANKER`, otherwise the baseline file below will be missing or incomplete.

In [ ]:
# Corpus-grounded GT from notebook 11 (defines which questions are answerable)
gt_df = pd.read_csv(DRIVE_PATH + "AWMF_CorpusGrounded_GroundTruth.csv")
gt_map = gt_df.set_index('English_Open_Question')['corpus_ground_truth'].to_dict()
answerable = {q for q, a in gt_map.items() if a != 'NOT_IN_CORPUS'}
work = df.reset_index(drop=True)   # ALL 200 questions
print(f"Generating UMLS on all {len(work)} MedQA questions  |  {len(answerable)} are corpus-answerable")

# Reuse notebook 11's baseline
BASELINE_FILE = DRIVE_PATH + "MISTRAL_rerank_full_results.json"
assert os.path.exists(BASELINE_FILE), (
    f"Baseline not found: {BASELINE_FILE}. Run notebook 11 first (DEV=False, "
    f"USE_RERANKER={USE_RERANKER}) so its baseline can be reused here.")
_b = json.load(open(BASELINE_FILE))
print(f"Reusing notebook-11 baseline: {len(_b['question'])} answers from {BASELINE_FILE}")
if len(_b['question']) < len(df):
    print(f"WARNING: baseline has only {len(_b['question'])}/{len(df)} answers -- finish notebook 11 for a full comparison.")

def run_umls():
    out_file = DRIVE_PATH + "UMLS_umls_results.json"
    if os.path.exists(out_file):
        res = json.load(open(out_file)); done = set(res["question"])
        print(f"  resuming -- {len(done)} already done")
    else:
        res = {"question":[],"answer":[],"contexts":[],"medqa_ground_truth":[],"corpus_ground_truth":[]}; done = set()
    for i in range(len(work)):
        r = work.iloc[i]; q = r['English_Open_Question']
        if q in done: continue
        try:
            ctx = retrieve_umls(q)
            ans = safe_invoke(mistral, qa_prompt.format(context="\n\n".join(ctx), question=q))
            res["question"].append(q); res["answer"].append(ans); res["contexts"].append(ctx)
            res["medqa_ground_truth"].append(r['English_Correct_Text'])
            res["corpus_ground_truth"].append(gt_map.get(q, "NOT_IN_CORPUS"))
            done.add(q)
            json.dump(res, open(out_file,"w"))
            if len(res["question"]) % 10 == 0: print(f"  [umls] {len(res['question'])}/{len(work)}")
            time.sleep(2)
        except Exception as e:
            print("err", i, str(e)[:100]); time.sleep(8)
    print(f"umls done -> {len(res['question'])}/{len(work)} -> {out_file}")
    return out_file

print("=== UMLS (baseline reused from notebook 11, not regenerated) ===")
run_umls()

Generating UMLS on all 200 MedQA questions  |  49 are corpus-answerable
Reusing notebook-11 baseline: 200 answers from /content/drive/MyDrive/MISTRAL_rerank_full_results.json
=== UMLS (baseline reused from notebook 11, not regenerated) ===
  [umls] 10/200
  [umls] 20/200
  [umls] 30/200
  [umls] 40/200
  [umls] 50/200
  [umls] 60/200
  [umls] 70/200
  [umls] 80/200
  [umls] 90/200
  [umls] 100/200
  [umls] 110/200
  [umls] 120/200
  [umls] 130/200
  [umls] 140/200
  [umls] 150/200
  [umls] 160/200
  [umls] 170/200
  [umls] 180/200
  [umls] 190/200
  [umls] 200/200
umls done -> 200/200 -> /content/drive/MyDrive/UMLS_umls_results.json


'/content/drive/MyDrive/UMLS_umls_results.json'

## 7. RAGAS evaluation --- baseline vs. UMLS, scored two ways

The same generated answers are scored under three views:
1. **Full 200 -- MedQA ground truth**: the whole benchmark, the headline number comparable to notebook 11.
2. **Answerable subset -- corpus-grounded ground truth**: the fair retrieval measure (questions the AWMF
   guidelines can actually answer; `NOT_IN_CORPUS` rows are excluded because there is no reference to score against).
3. **Answerable subset -- MedQA ground truth**: a parity check, the MedQA number on the same subset as view 2.

The judge runs with `max_tokens=4096` to avoid the truncation error.

In [ ]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))   # avoids judge truncation
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})

def score(path, gt_key, answerable_only=False):
    d = json.load(open(path))
    keep = list(range(len(d["question"])))
    # corpus GT only exists for answerable questions; drop NOT_IN_CORPUS rows when scoring against it.
    if gt_key == "corpus_ground_truth" or answerable_only:
        keep = [i for i in keep if d["corpus_ground_truth"][i] != "NOT_IN_CORPUS"]
    dd = {"question":[d["question"][i] for i in keep], "answer":[d["answer"][i] for i in keep],
          "contexts":[d["contexts"][i] for i in keep], "ground_truth":[d[gt_key][i] for i in keep]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    means = out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3)
    return dict(means), len(keep)

BASE = BASELINE_FILE                              # notebook-11 baseline (reused, not regenerated)
UMLS = DRIVE_PATH + "UMLS_umls_results.json"      # generated above
def show(label, path, gt_key, answerable_only=False):
    m, n = score(path, gt_key, answerable_only)
    print(f"{label:9s} (n={n}):", m)

print("=== FULL 200 -- MedQA ground truth (the whole benchmark) ===")
show("BASELINE", BASE, "medqa_ground_truth"); show("UMLS", UMLS, "medqa_ground_truth")
print("\n=== ANSWERABLE SUBSET -- corpus-grounded ground truth (fair retrieval measure) ===")
show("BASELINE", BASE, "corpus_ground_truth"); show("UMLS", UMLS, "corpus_ground_truth")
print("\n=== ANSWERABLE SUBSET -- MedQA ground truth (parity check) ===")
show("BASELINE", BASE, "medqa_ground_truth", True); show("UMLS", UMLS, "medqa_ground_truth", True)

/tmp/ipykernel_5244/827877507.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_5244/827877507.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_5244/827877507.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_precision

=== FULL 200 -- MedQA ground truth (the whole benchmark) ===


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[22]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[782]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


BASELINE  (n=200): {'context_precision': np.float64(0.189), 'context_recall': np.float64(0.23), 'faithfulness': np.float64(0.536), 'answer_relevancy': np.float64(0.261)}


...

UMLS      (n=49): {'context_precision': np.float64(0.604), 'context_recall': np.float64(0.794), 'faithfulness': np.float64(0.545), 'answer_relevancy': np.float64(0.441)}

=== ANSWERABLE SUBSET -- MedQA ground truth (parity check) ===


Evaluating:   0%|          | 0/196 [00:00<?, ?it/s]

BASELINE  (n=49): {'context_precision': np.float64(0.389), 'context_recall': np.float64(0.367), 'faithfulness': np.float64(0.65), 'answer_relevancy': np.float64(0.509)}


Evaluating:   0%|          | 0/196 [00:00<?, ?it/s]

UMLS      (n=49): {'context_precision': np.float64(0.366), 'context_recall': np.float64(0.32), 'faithfulness': np.float64(0.542), 'answer_relevancy': np.float64(0.453)}


## 8. Summary Comparison Table

Based on the execution results, here is the performance comparison across the three evaluation views:

### Full 200 -- MedQA Ground Truth
| Method | n | Context Precision | Context Recall | Faithfulness | Answer Relevancy |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Baseline** | 200 | 0.189 | 0.230 | 0.536 | 0.261 |
| **UMLS** | 200 | 0.142 | 0.158 | 0.516 | 0.216 |

### Answerable Subset -- Corpus-Grounded Ground Truth
| Method | n | Context Precision | Context Recall | Faithfulness | Answer Relevancy |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Baseline** | 49 | 0.671 | 0.797 | 0.639 | 0.517 |
| **UMLS** | 49 | 0.604 | 0.794 | 0.545 | 0.441 |

### Answerable Subset -- MedQA Ground Truth (Parity Check)
| Method | n | Context Precision | Context Recall | Faithfulness | Answer Relevancy |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Baseline** | 49 | 0.389 | 0.367 | 0.650 | 0.509 |
| **UMLS** | 49 | 0.366 | 0.320 | 0.542 | 0.453 |

**Observations:**
In this specific configuration (Mistral, no reranker), the free-form LLM expansion (Baseline) consistently outperformed the UMLS ontology-guided expansion across all metrics. This suggests that while UMLS provides formal synonyms, the LLM's broader context or translation capability might be more effective for this specific cross-lingual retrieval task on the MedQA benchmark.